Import Dependencies

In [40]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

Download All data from Qdrant client

In [41]:
qdrant_client=QdrantClient(url="http://localhost:6333")

In [42]:
all_points=qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False,
)

In [43]:
all_points[0][0].payload
                

{'preprocessed_description': "Counlisha Real Wood Stand for Echo Dots(4th Gen)(5th Gen),Tripod Accessories Protect Smart Speaker get Better Sound,Secure Stable Wooden Mount Holder for Echo Dot (Color:Walnut) 👉【Compatibility】Only be applicable for Dot(4th Gen) and homepod mini speaker ,the wood Leg , metal frame and soft foam hold the speaker well, make the E ch0 Dot(4th Gen)to stay firmly on the stand , don't worry about slip off and fall out in using. 👉【Protection Function】We added anti-slip silicone pads to the wooden legs so that the holder did not slide easily, Then hold the speaker up without any water affection on the table. 👉【Unique Design】Not worry about any discordant;The wooden stand with right size can match with Dots(4th Gen) well, it's a simply and stylish design to use for office, home, garden etc. it's good for sound and easy to install; 👉【Real wood materials】The wood stand is made of walnut from North America, it's 100% real wood and coating by oil from nature. the blac

In [44]:
all_points[0]



[Record(id=0, payload={'preprocessed_description': "Counlisha Real Wood Stand for Echo Dots(4th Gen)(5th Gen),Tripod Accessories Protect Smart Speaker get Better Sound,Secure Stable Wooden Mount Holder for Echo Dot (Color:Walnut) 👉【Compatibility】Only be applicable for Dot(4th Gen) and homepod mini speaker ,the wood Leg , metal frame and soft foam hold the speaker well, make the E ch0 Dot(4th Gen)to stay firmly on the stand , don't worry about slip off and fall out in using. 👉【Protection Function】We added anti-slip silicone pads to the wooden legs so that the holder did not slide easily, Then hold the speaker up without any water affection on the table. 👉【Unique Design】Not worry about any discordant;The wooden stand with right size can match with Dots(4th Gen) well, it's a simply and stylish design to use for office, home, garden etc. it's good for sound and easy to install; 👉【Real wood materials】The wood stand is made of walnut from North America, it's 100% real wood and coating by oil

In [45]:
all_context=[{"id":data.payload["parent_asin"],"text":data.payload["preprocessed_description"]}  for data in all_points[0]]

In [46]:
all_context

[{'id': 'B0BBNCNPPF',
  'text': "Counlisha Real Wood Stand for Echo Dots(4th Gen)(5th Gen),Tripod Accessories Protect Smart Speaker get Better Sound,Secure Stable Wooden Mount Holder for Echo Dot (Color:Walnut) 👉【Compatibility】Only be applicable for Dot(4th Gen) and homepod mini speaker ,the wood Leg , metal frame and soft foam hold the speaker well, make the E ch0 Dot(4th Gen)to stay firmly on the stand , don't worry about slip off and fall out in using. 👉【Protection Function】We added anti-slip silicone pads to the wooden legs so that the holder did not slide easily, Then hold the speaker up without any water affection on the table. 👉【Unique Design】Not worry about any discordant;The wooden stand with right size can match with Dots(4th Gen) well, it's a simply and stylish design to use for office, home, garden etc. it's good for sound and easy to install; 👉【Real wood materials】The wood stand is made of walnut from North America, it's 100% real wood and coating by oil from nature. the b

Render a prompt to generate synthetic Eval refernce dataset

In [47]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}


SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

In [48]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, 

In [49]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[
  {
    "id": "B0BBNCNPPF",
    "text": "Counlisha Real Wood Stand for Echo Dots(4th Gen)(5th Gen),Tripod Accessories Protect Smart Speaker get Better Sound,Secure Stable Wooden Mount Holder for Echo Dot (Color:Walnut) \ud83d\udc49\u3010Compatibility\u3011Only be applicable for Dot(4th Gen) and homepod mini speaker ,the wood Leg , metal frame and soft foam hold the speaker well, make the E ch0 Dot(4th Gen)to stay firmly on the stand , don't worry about slip off and fall out in using. \ud83d\udc49\u3010Protection Function\u3011We added anti-slip silicone pads to the wooden legs so that the holder did not slide easily, Then hold the speaker up without any water affection on the table. \ud83d\udc49\u3010Unique Design\u3011Not worry about any discordant;The wooden stand with right size can match with Dots(4th Gen) well, it's a simply and stylish design to use for office, home, garden etc. it's good for sound

In [50]:
response = openai.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

[
  {
    "reasoning": "This can be answered from one product because the item description directly states the compatibility, materials, stability features, and what is included in the box.",
    "question": "Will this wooden stand fit my Echo Dot 4th generation?",
    "chunk_ids": [
      "B0BBNCNPPF"
    ],
    "answer_example": "Yes, this stand is compatible with the Echo Dot 4th Gen. It is designed to hold the speaker securely and help prevent slipping.",
    "reasoning": "The product text explicitly says it is applicable to Dot 4th Gen and describes the stand’s protective and compatibility features."
  },
  {
    "reasoning": "This can be answered from one product because the antenna listing gives a clear install time and setup requirement.",
    "question": "How hard is it to install this TV antenna?",
    "chunk_ids": [
      "B09R839VRC"
    ],
    "answer_example": "It is very easy to install. The listing says no assembly is required and setup takes less than 3 minutes.",
    

In [51]:
response.usage

CompletionUsage(completion_tokens=3717, prompt_tokens=19269, total_tokens=22986, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

In [52]:
json_output = response.choices[0].message.content

In [53]:
json_output


'[\n  {\n    "reasoning": "This can be answered from one product because the item description directly states the compatibility, materials, stability features, and what is included in the box.",\n    "question": "Will this wooden stand fit my Echo Dot 4th generation?",\n    "chunk_ids": [\n      "B0BBNCNPPF"\n    ],\n    "answer_example": "Yes, this stand is compatible with the Echo Dot 4th Gen. It is designed to hold the speaker securely and help prevent slipping.",\n    "reasoning": "The product text explicitly says it is applicable to Dot 4th Gen and describes the stand’s protective and compatibility features."\n  },\n  {\n    "reasoning": "This can be answered from one product because the antenna listing gives a clear install time and setup requirement.",\n    "question": "How hard is it to install this TV antenna?",\n    "chunk_ids": [\n      "B09R839VRC"\n    ],\n    "answer_example": "It is very easy to install. The listing says no assembly is required and setup takes less than 

In [54]:
json_output = json_output.replace("```json", "").replace("```", "")

In [55]:
print(json_output)

[
  {
    "reasoning": "This can be answered from one product because the item description directly states the compatibility, materials, stability features, and what is included in the box.",
    "question": "Will this wooden stand fit my Echo Dot 4th generation?",
    "chunk_ids": [
      "B0BBNCNPPF"
    ],
    "answer_example": "Yes, this stand is compatible with the Echo Dot 4th Gen. It is designed to hold the speaker securely and help prevent slipping.",
    "reasoning": "The product text explicitly says it is applicable to Dot 4th Gen and describes the stand’s protective and compatibility features."
  },
  {
    "reasoning": "This can be answered from one product because the antenna listing gives a clear install time and setup requirement.",
    "question": "How hard is it to install this TV antenna?",
    "chunk_ids": [
      "B09R839VRC"
    ],
    "answer_example": "It is very easy to install. The listing says no assembly is required and setup takes less than 3 minutes.",
    

In [56]:
json_output = json.loads(json_output)

In [57]:
json_output

[{'reasoning': 'The product text explicitly says it is applicable to Dot 4th Gen and describes the stand’s protective and compatibility features.',
  'question': 'Will this wooden stand fit my Echo Dot 4th generation?',
  'chunk_ids': ['B0BBNCNPPF'],
  'answer_example': 'Yes, this stand is compatible with the Echo Dot 4th Gen. It is designed to hold the speaker securely and help prevent slipping.'},
 {'reasoning': 'The antenna product clearly states installation is simple and fast, with no extra product needed.',
  'question': 'How hard is it to install this TV antenna?',
  'chunk_ids': ['B09R839VRC'],
  'answer_example': 'It is very easy to install. The listing says no assembly is required and setup takes less than 3 minutes.'},
 {'reasoning': 'All requested features are stated directly in the earbud product title.',
  'question': 'Do these wireless earbuds have a microphone and are they waterproof?',
  'chunk_ids': ['B0C2TLBSMP'],
  'answer_example': 'Yes, they include a microphone a

In [58]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

In [59]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

Single chunk questions: 17
Multiple chunk questions: 8
Impossible questions: 5


In [60]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B09R839VRC")
            )
        ]
    )
)[0]

In [61]:
points[0].payload

{'preprocessed_description': "TV Antenna, 2023 Newest HDTV Indoor Digital TV Antenna 300 Miles Range with Amplifier Signal Booster 4K HD Free Local Channels Support All Television - 10ft High Performance Coax Cable ✔Enjoy Free Channels: Enjoy Free lifetime 4k an HD channel with TV indoor antenna. By picking up most broadcasting signals in your area, you don’t have to worry about monthly expensive cable plans and you can save $1020 per year. ✔300 Miles Long Range Antenna: 2023 Update Amplifier Signal Booster, 300 Miles signal reception range, equip with new type switch control amplifier booster. Below 35 miles, Turn to the short range side! Above 35 miles, Turn the green light on. ✔10FT Easiest To Install: Setup is a breeze — no assembly required, you can easily install it in less than 3 minutes and 10FT long cable makes it easy to place the antenna in the best reception spot in your home. ✔Compact And Small: Don't let the size discourage you! It's tiny but mighty; It doesn't take up a 

In [62]:
def get_description(parent_asin: str) -> str:
    
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]

    return points[0].payload["preprocessed_description"]

In [64]:
get_description("B09R839VRC")

"TV Antenna, 2023 Newest HDTV Indoor Digital TV Antenna 300 Miles Range with Amplifier Signal Booster 4K HD Free Local Channels Support All Television - 10ft High Performance Coax Cable ✔Enjoy Free Channels: Enjoy Free lifetime 4k an HD channel with TV indoor antenna. By picking up most broadcasting signals in your area, you don’t have to worry about monthly expensive cable plans and you can save $1020 per year. ✔300 Miles Long Range Antenna: 2023 Update Amplifier Signal Booster, 300 Miles signal reception range, equip with new type switch control amplifier booster. Below 35 miles, Turn to the short range side! Above 35 miles, Turn the green light on. ✔10FT Easiest To Install: Setup is a breeze — no assembly required, you can easily install it in less than 3 minutes and 10FT long cable makes it easy to place the antenna in the best reception spot in your home. ✔Compact And Small: Don't let the size discourage you! It's tiny but mighty; It doesn't take up a lot of space,perfect to blend

Create Eval dataset in LangSmith

In [65]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [70]:
ls_client = Client()

In [77]:
dataset_name = "rag-evaluation-dataset"

if ls_client.has_dataset(dataset_name=dataset_name):
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
else:
    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description="RAG evaluation dataset"
    )

In [78]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )